# 1.1.5 Data Structures

Lists, dicts, sets, tuples, Counter — applied to MovieLens ratings data.

Dataset: MovieLens Small from HuggingFace

In [ ]:
import urllib.request, csv
from pathlib import Path
from collections import defaultdict, Counter

DATA = Path('data'); DATA.mkdir(exist_ok=True)
for name, url in [
    ('ratings.csv', 'https://huggingface.co/datasets/Shahrukh0/MovieLens-Small/resolve/main/ratings.csv'),
    ('movies.csv',  'https://huggingface.co/datasets/Shahrukh0/MovieLens-Small/resolve/main/movies.csv'),
]:
    dest = DATA / name
    if not dest.exists():
        try: urllib.request.urlretrieve(url, dest); print(f'Downloaded {name}')
        except Exception as e: print(f'Failed {name}: {e}')

with open(DATA/'ratings.csv') as f: ratings = list(csv.DictReader(f))
with open(DATA/'movies.csv')  as f: movies  = list(csv.DictReader(f))
print(f'Ratings: {len(ratings):,}, Movies: {len(movies):,}')

In [ ]:
# Dict: movie lookup + aggregated stats
movie_map   = {m['movieId']: m['title'] for m in movies}
movie_rt    = defaultdict(list)
for r in ratings:
    movie_rt[r['movieId']].append(float(r['rating']))

stats = {mid: {'title': movie_map.get(mid, mid), 'mean': sum(rs)/len(rs), 'n': len(rs)}
         for mid, rs in movie_rt.items() if len(rs) >= 20}

import pandas as pd
stats_df = pd.DataFrame(stats).T.sort_values('mean', ascending=False)
stats_df['mean'] = stats_df['mean'].astype(float).round(2)
stats_df.head(10)

In [ ]:
# Sets: genre analysis
all_genres = set()
for m in movies:
    for g in m.get('genres','').split('|'):
        if g and g != '(no genres listed)': all_genres.add(g)
print(f'Unique genres: {sorted(all_genres)}')
print(f'Total: {len(all_genres)}')

In [ ]:
# Counter: rating distribution
import matplotlib.pyplot as plt
score_counts = Counter(float(r['rating']) for r in ratings)
scores, counts = zip(*sorted(score_counts.items()))
plt.bar(scores, counts, width=0.4, color='steelblue', edgecolor='white')
plt.xlabel('Rating'); plt.ylabel('Count'); plt.title('Rating Distribution (MovieLens)')
plt.tight_layout(); plt.show()

In [ ]:
# Top-N movies by rating count (list operations)
top_n = sorted(stats.values(), key=lambda x: x['n'], reverse=True)[:10]
for s in top_n:
    print(f"  {s['title'][:40]:<40}  n={s['n']:>5}  mean={float(s['mean']):.2f}")